# Lábia — treino SFT do aluno (`sem-filtro`)

Este notebook treina o **Qwen3-4B-Instruct-2507** com LoRA. Variante: a **ablação**: o mesmo número de conversas, sorteadas entre todas as do professor, aprovadas ou não. Se este modelo sair tão bom quanto o principal, o filtro do verificador não fez diferença.

Etapas:

1. junta as duas fatias da geração (anexadas como fonte deste notebook);
2. mede a velocidade das configurações de treino aqui mesmo e treina com a mais rápida;
3. **funde** o adaptador LoRA nos pesos do modelo: o resultado é um modelo único, que não depende mais do adaptador;
4. converte para **GGUF de 8 bits**, o formato que a avaliação (Ollama) e a demo (CPU) usam.

O treino para sozinho ao bater no orçamento de 8 horas e salva o que aprendeu até ali.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
LABIA_COMMIT = "ab3c24ec6dbc0739b4f414b5dfb08aa5c3fea9c7"
VARIANT = "sem-filtro"
DATA_FILE = "sft-sem-filtro.jsonl"
TRAIN_CMD = ["python", "-m", "labia.train_sft"]
EPOCHS = "1.33"
BUDGET_H = "8"
BASE = "Qwen/Qwen3-4B-Instruct-2507"
WORK = Path("/kaggle/working")
ADAPTER = WORK / f"adaptador-{VARIANT}"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes",
                    f"git+https://github.com/deniscarvalho-tech/labia.git@{LABIA_COMMIT}"], capture_output=True, text=True)
print((r.stdout + r.stderr)[-400:])
if "--unsloth" in TRAIN_CMD:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], capture_output=True, text=True)
    print((r.stdout + r.stderr)[-400:])

## 1. Juntar as fatias da geração

In [ ]:
parts = sorted(Path("/kaggle/input").rglob(DATA_FILE))
print([str(p) for p in parts])
assert len(parts) == 2, "esperava as duas fatias da geração anexadas como fonte"
rows = [line for p in parts for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]
data = WORK / "treino.jsonl"
data.write_text("\n".join(rows) + "\n", encoding="utf-8")
from collections import Counter
meta = [json.loads(r)["meta"] for r in rows]
print(len(rows), "conversas;", dict(Counter(m["tactic"] for m in meta)), dict(Counter("legitimo" if m["legit"] else "golpe" for m in meta)))

## 2. A mesma configuração do treino principal

Esta é uma **ablação**: para a única diferença entre os dois modelos ser o filtro do verificador, o treino usa exatamente a configuração que o treino principal mediu e escolheu (`unsloth-qlora-1gpu`) e o mesmo número de épocas (1.33). Por isso não há sonda de velocidade aqui.

In [ ]:
# Mesma ordem do treino principal: tira o torchao e só depois instala o Unsloth, se a configuração usar.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True)  # a 0.10 do Kaggle quebra o LoRA da peft
TRAIN_CMD = [sys.executable if c == "PYTHON" else c for c in ["PYTHON", "-m", "labia.train_sft", "--unsloth", "--mode", "qlora"]]
TRAIN_ENV = {"CUDA_VISIBLE_DEVICES": "0"}
EPOCHS = "1.33"
if "--unsloth" in TRAIN_CMD:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], capture_output=True, text=True)
    print((r.stdout + r.stderr)[-300:])
print("configuração fixa:", "unsloth-qlora-1gpu", "| épocas:", EPOCHS, "|", " ".join(TRAIN_CMD))

## 3. Treinar

In [ ]:
start = time.perf_counter()
cmd = TRAIN_CMD + ["--data", str(data), "--out", str(ADAPTER), "--epochs", EPOCHS, "--time-budget-h", BUDGET_H]
print(" ".join(cmd))
p = subprocess.run(cmd, capture_output=True, text=True, env=dict(os.environ, **TRAIN_ENV))
print(p.stdout[-3000:])
if p.returncode != 0:
    print(p.stderr[-4000:])
    raise SystemExit("o treino falhou")
print(f"treino em {(time.perf_counter() - start) / 3600:.2f} h")
print(json.dumps({k: v for k, v in json.loads((ADAPTER / "treino.json").read_text()).items() if k != "log"}, indent=1))

## 4. Fundir o adaptador e converter para GGUF de 8 bits

A fusão roda na CPU (o modelo em float16 ocupa 8 GB de RAM; a máquina tem 31 GB) para não disputar a GPU. A conversão usa o script oficial do llama.cpp.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

merged = Path("/tmp/merged")
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map={"": "cpu"})
model = PeftModel.from_pretrained(model, str(ADAPTER)).merge_and_unload()
model.save_pretrained(merged, safe_serialization=True)
AutoTokenizer.from_pretrained(BASE).save_pretrained(merged)
del model
subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ggml-org/llama.cpp", "/tmp/llama.cpp"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/tmp/llama.cpp/gguf-py", "sentencepiece"], check=True)
gguf = WORK / f"labia-{VARIANT}-q8_0.gguf"
subprocess.run([sys.executable, "/tmp/llama.cpp/convert_hf_to_gguf.py", str(merged), "--outtype", "q8_0", "--outfile", str(gguf)],
               check=True, stdout=subprocess.DEVNULL)
print(gguf.name, f"{gguf.stat().st_size / 1024**3:.2f} GB")
for f in sorted(WORK.iterdir()):
    print(f.name)